In [1]:
from langgraph.graph import StateGraph,START,END,MessagesState 
from langgraph.checkpoint.memory import InMemorySaver 
from langchain_core.messages import BaseMessage,HumanMessage,AIMessage 


In [2]:
from dotenv import load_dotenv 
load_dotenv()

True

In [3]:
from langchain_groq import ChatGroq
llm=ChatGroq(model='Llama-3.3-70b-Versatile')


In [4]:
checkpointer=InMemorySaver()

In [5]:
#lets design the graph for chatting with the llm 

graph=StateGraph(MessagesState)

In [6]:
def chat_with_llm(state: MessagesState)-> MessagesState:
    
    response=llm.invoke(state['messages']) 
    
    return {'messages':[response]}

In [7]:
graph.add_node('chat',chat_with_llm)

graph.add_edge(START,'chat') 
graph.add_edge('chat',END)

In [8]:
workflow=graph.compile(checkpointer=checkpointer)

In [9]:
config={'configurable':{'thread_id':'32'}} 

In [10]:
initial_state={'messages': [HumanMessage(content="Hi, My name is aashish")]}
# initial_state={'messages':[HumanMessage(content='Do you know my name')]}

In [11]:
final_state=workflow.invoke(input=initial_state,config=config)

In [ ]:
final_state

{'messages': [HumanMessage(content='Hi, My name is aashish', additional_kwargs={}, response_metadata={}, id='6ce1ce0e-e01b-4450-a28a-00dab6edae28'),
  AIMessage(content="Hello Aashish, it's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 43, 'total_tokens': 71, 'completion_time': 0.041930837, 'prompt_time': 0.003936843, 'queue_time': 0.061071427, 'total_time': 0.04586768}, 'model_name': 'Llama-3.3-70b-Versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--4a2780d9-fb51-428e-b632-5106c110165b-0', usage_metadata={'input_tokens': 43, 'output_tokens': 28, 'total_tokens': 71})]}

In [13]:
#This is simple short term memory which remains for the current execution only

In [14]:
#if we wants that it doesn't gets deleted after the execution we have to use the dbms persistence